# F1 Rule & Penalty Interpreter

**Team Members:** Hrishi Kabra & Ayush Bhatia

## Project Overview

Our project is an AI powered assistant designed to help Formula 1 fans understand the complex FIA regulations and steward decisions. Penalties and rules in Formula 1 are quite difficult to interpret and often span across long and cross referenced rulebooks. By using a RAG pipeline, our system can ingest FIA sporting regulations and Steward Decision documents to translate these legal eplanations into plain English. 

Both of us are passionate Formula 1 fans and we are building this to help watchers understand these complicated rules. 

## Current Status
We have a working MVP that successfully ingests the 2025 F1 regulations and decision document pdfs into a knowledge base. It routes the questions, retrieves the relevant passages and uses the OpenAI api key to help create answers. The gradio app is working and can answer questions through rag and can identify f1 documents. 

Current failure - Our chunk size is set to 300 - because of this the semantic search matches the introductory headers of the steward decision docs rather than the actual reasoning below - causes llm to hallucinate the penalty justifications - loses context.

## 1) Setup


In [1]:
# Install lightweight dependencies.
# Keep installs minimal for student reliability.
!pip -q install gradio pandas numpy pypdf openai

import os, re, json, time, math, hashlib
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import numpy as np

# Create folders for logs/results (safe if they already exist)
os.makedirs("runs", exist_ok=True)
os.makedirs("data", exist_ok=True)


In [2]:
# Standard setup cell
# Clones your specific GitHub repository
!git clone --depth 1 -q https://github.com/HrishiKabra/AI_Engineering_Project.git /content/project_repo

!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git /content/main_course_repo
import sys; sys.path.append('/content/main_course_repo')
from course_utils import lab5_setup, get_text_embedding
lab5_setup()


fatal: destination path '/content/project_repo' already exists and is not an empty directory.
fatal: destination path '/content/main_course_repo' already exists and is not an empty directory.
🔧 Setting up your environment...
  → Installing core packages...
  → Setting random seed for reproducible results...
  → Checking API key...
🔑 Enter your OpenAI API key.
   (It will only be stored in this Colab runtime - it's safe!)
   Get your key from: https://platform.openai.com/api-keys
✅ API key set.
  → Adding course files to path...
✅ Setup complete!
✅ lab5_setup complete — ready.


In [3]:
from pathlib import Path
from pypdf import PdfReader

def pdf_to_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        txt = page.extract_text() or ""
        parts.append(txt)
    return "\n".join(parts)

def load_pdfs(folder: Path):
    docs = []
    for p in sorted(folder.rglob("*.pdf")):
        text = pdf_to_text(p).strip()
        if len(text) > 500:  # avoid tiny/empty extracts
            docs.append({"source": str(p), "text": text})
    return docs

## 2) Configuration


In [4]:
@dataclass
class Config:
    # LLM
    model: str = "mock-model"
    temperature: float = 0.2
    max_tokens: int = 400
    use_mock_llm: bool = False  # ✅ set False when you wire a real API

    # RAG
    use_rag: bool = True
    chunk_size: int = 300
    chunk_overlap: int = 50
    top_k: int = 4

    # Tools / Router
    use_router: bool = True

    # Safety & trust
    guardrails_on: bool = True
    require_citations: bool = True
    low_confidence_threshold: float = 0.25  # heuristic for warnings

    # Logging / Ops
    log_path: str = "runs/interactions.jsonl"

cfg = Config()
cfg


Config(model='mock-model', temperature=0.2, max_tokens=400, use_mock_llm=False, use_rag=True, chunk_size=300, chunk_overlap=50, top_k=4, use_router=True, guardrails_on=True, require_citations=True, low_confidence_threshold=0.25, log_path='runs/interactions.jsonl')

## 3) Data (Knowledge Base)
Our data consists of official FIA Formula One Sporting Regulations and Steward Decisions from the 2025/2026 seasons.
- **Sources:** [FIA Regulations](https://www.fia.com/regulation/category/110) & [FIA Documents](https://www.fia.com/documents/)


In [5]:
from pathlib import Path
# Point to the cloned repository folder
BASE = Path("/content/project_repo/docs")
REGS = BASE / "regulations"
DECS = BASE / "decision_docs"

reg_docs = load_pdfs(REGS)
dec_docs = load_pdfs(DECS)
docs = reg_docs + dec_docs

print("regulations:", len(reg_docs))
print("decisions:", len(dec_docs))
print("total:", len(docs))
if docs:
    print("example source:", docs[0]["source"])
    print(docs[0]["text"][:800])
else:
    print("No PDFs found. Make sure the github repo URL is correct and contains the docs folder.")


regulations: 2
decisions: 17
total: 19
example source: /content/project_repo/docs/regulations/fia_2025_f1_guidelines_penalty_points_overview.pdf
2025 FORMULA 1 PENALTY AND POINT GUIDELINES  
VERSION - 14 MAY 2025 
 
    
1 
Penalty 
Guidelines 
1 
 
Guidelines for Penalties and Points – 2025 [14 MAY] [CLEAN] 
 
 
 
 
Offence SR Article Free Practice PP Qualifying PP Race PP 
Sanctions 
Receiving five reprimands, four of which were imposed for a driving 
infringement 
18.2 Ten grid places 0 Ten grid places 0 Ten grid places (mandatory) 0 
Personnel Curfew and Covers On Breaches 
Breach of Personnel Curfew 23.6 & 23.5     Pit lane start for both 
Competitor cars (mandatory) 
0 
Breach of “Covers On” Time 23.6 & 38.2a) i)     Pit lane start for both 
Competitor cars (mandatory) 
 
Receiving Mechanical Assistance 
Re-joining by use of mechanical assistance 26.4     Disqualification 0 
Use of Tyres 
Use of tyres without appropriate iden


In [6]:
KB = []

def add_folder_to_kb(folder, prefix):
    for pdf in folder.glob("*.pdf"):
        text = pdf_to_text(pdf)

        if len(text) < 200:
            continue

        KB.append({
            "id": f"{prefix}_{pdf.stem}",
            "title": pdf.stem,
            "text": text,
            "source": str(pdf)
        })

add_folder_to_kb(REGS, "regulation")
add_folder_to_kb(DECS, "decision")

print("documents loaded:", len(KB))
print(KB[0]["title"])
print(KB[0]["text"][:500])

documents loaded: 20
fia_2025_f1_guidelines_penalty_points_overview
2025 FORMULA 1 PENALTY AND POINT GUIDELINES  
VERSION - 14 MAY 2025 
 
    
1 
Penalty 
Guidelines 
1 
 
Guidelines for Penalties and Points – 2025 [14 MAY] [CLEAN] 
 
 
 
 
Offence SR Article Free Practice PP Qualifying PP Race PP 
Sanctions 
Receiving five reprimands, four of which were imposed for a driving 
infringement 
18.2 Ten grid places 0 Ten grid places 0 Ten grid places (mandatory) 0 
Personnel Curfew and Covers On Breaches 
Breach of Personnel Curfew 23.6 & 23.5     Pit lane start fo


### 3.1) Document Chunking


In [7]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    # Simple character-based chunker (good enough to start)
    if chunk_size <= 0:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

def build_chunks(kb: List[Dict[str, Any]], cfg: Config) -> List[Dict[str, Any]]:
    chunks = []
    for doc in kb:
        for i, chunk in enumerate(chunk_text(doc["text"], cfg.chunk_size, cfg.chunk_overlap)):
            chunks.append({
                "chunk_id": f'{doc["id"]}::chunk{i}',
                "doc_id": doc["id"],
                "title": doc["title"],
                "text": chunk,
                "source": doc.get("source", ""),
            })
    return chunks

CHUNKS = build_chunks(KB, cfg)
len(CHUNKS), CHUNKS[0]["chunk_id"]


(1314, 'regulation_fia_2025_f1_guidelines_penalty_points_overview::chunk0')

## 4) Core Components
### Logging


In [8]:
def now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def append_jsonl(path: str, record: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def stable_hash(obj: Any) -> str:
    raw = json.dumps(obj, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:12]


### LLM Wrapper


In [9]:
from openai import OpenAI
import os

# --- TODO: Add your OpenAI API Key below or use Colab Secrets ---
# os.environ['OPENAI_API_KEY'] = 'sk-...'
client = OpenAI() # It will automatically pick up the OPENAI_API_KEY from environment

def call_llm_real_api(prompt: str, cfg: Config) -> Dict[str, Any]:
    """
    Calls the real OpenAI API.
    """
    # Default to gpt-4o-mini if 'mock-model' is still in config
    model_name = cfg.model if cfg.model != 'mock-model' else 'gpt-4o-mini'
    
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful AI assistant tasked with answering questions about Formula 1 regulations and decisions."}, 
                {"role": "user", "content": prompt}
            ],
            temperature=cfg.temperature,
            max_tokens=cfg.max_tokens,
        )
        text = response.choices[0].message.content
        
        # Optional usage parsing
        usage = {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }
    except Exception as e:
        text = f"Error calling OpenAI API: {e}"
        usage = {"prompt_tokens": None, "completion_tokens": None, "total_tokens": None}
        
    return {
        "text": text,
        "usage": usage,
        "model": model_name,
        "latency_ms": 0.0,
        "cost_usd": 0.0
    }

def call_llm_mock(prompt: str, cfg: Config) -> Dict[str, Any]:
    response = (
        "MOCK ANSWER:\n"
        f"- I received your question/prompt: {prompt[:180]!r}\n"
        "- In a real project, this would be answered using the selected tool(s) and/or RAG.\n"
        "- Now providing a concise response placeholder."
    )
    return {
        "text": response,
        "usage": {"prompt_tokens": None, "completion_tokens": None, "total_tokens": None},
        "model": cfg.model,
        "latency_ms": 0.5,
        "cost_usd": 0.0
    }

import time
def call_llm(prompt: str, cfg: Config) -> Dict[str, Any]:
    t0 = time.time()
    if getattr(cfg, 'use_mock_llm', False):
        out = call_llm_mock(prompt, cfg)
    else:
        out = call_llm_real_api(prompt, cfg)
    
    out['latency_ms'] = round((time.time() - t0) * 1000, 2)
    return out


### Retrieval System


In [10]:
def keyword_score(query: str, text: str) -> float:
    q = re.findall(r"[a-zA-Z0-9']+", query.lower())
    t = re.findall(r"[a-zA-Z0-9']+", text.lower())
    if not q or not t:
        return 0.0
    qset = set(q)
    hits = sum(1 for w in t if w in qset)
    return hits / max(1, len(t))

def retrieve(query: str, chunks: List[Dict[str, Any]], cfg: Config) -> Tuple[List[Dict[str, Any]], float]:
    scored = []
    for c in chunks:
        s = keyword_score(query, c["text"] + " " + c["title"])
        scored.append((s, c))
    scored.sort(key=lambda x: x[0], reverse=True)

    top = [c for s, c in scored[:cfg.top_k] if s > 0]
    conf = float(min(1.0, scored[0][0])) if scored else 0.0
    return top, conf


### Tools


In [11]:
def safe_calculator(expr: str) -> str:
    if len(expr) > 80:
        return "Expression too long."
    if re.search(r"[A-Za-z_]|\.|\[|\]|\{|\}|;|:", expr):
        return "Unsupported expression."
    try:
        val = eval(expr, {"__builtins__": {}}, {})
        if isinstance(val, (int, float)) and (not math.isfinite(val) or abs(val) > 1e12):
            return "Result out of range."
        return str(val)
    except Exception:
        return "Could not evaluate expression."

def tool_rag_search(query: str, cfg: Config) -> Dict[str, Any]:
    hits, conf = retrieve(query, CHUNKS, cfg)
    return {"hits": hits, "confidence": conf}

def tool_calculate(query: str, cfg: Config) -> Dict[str, Any]:
    return {"result": safe_calculator(query)}


### Safety Guardrails


In [12]:
UNSAFE_TERMS = [
    "how to build a bomb",
    "make a weapon",
]

def guardrail_check_input(user_input: str) -> Optional[str]:
    text = user_input.lower().strip()
    if len(text) > 2000:
        return "I can't process inputs that long in this demo. Please shorten your question."
    for term in UNSAFE_TERMS:
        if term in text:
            return "I can't help with that request."
    return None

def strip_prompt_injection(retrieved_text: str) -> str:
    lines = retrieved_text.splitlines()
    cleaned = []
    for ln in lines:
        if re.search(r"(ignore previous|system prompt|you are chatgpt|developer message)", ln.lower()):
            continue
        cleaned.append(ln)
    return "\n".join(cleaned)

def guardrail_check_output(answer: str) -> Optional[str]:
    # TODO: add domain-specific output rules, e.g. no medical/legal advice.
    return None


### Pipeline Execution


In [13]:
def build_answer_prompt(user_input: str, retrieved: List[Dict[str, Any]]) -> str:
    sources_block = ""
    if retrieved:
        formatted = []
        for i, c in enumerate(retrieved, start=1):
            snippet = strip_prompt_injection(c["text"])
            formatted.append(f"[{i}] {c['title']}: {snippet}")
        sources_block = "\n\nSOURCES:\n" + "\n\n".join(formatted)

    return (
        "You are a helpful assistant.\n"
        "If you use sources, cite them like [1], [2].\n"
        "If you are unsure, say so and ask a follow-up question.\n\n"
        f"USER QUESTION:\n{user_input}\n"
        f"{sources_block}\n\n"
        "ANSWER:"
    )


In [14]:
@dataclass
class PipelineResult:
    answer: str
    sources: List[Dict[str, Any]]
    tool_trace: List[str]
    warnings: List[str]
    metadata: Dict[str, Any]

def route(user_input: str, cfg: Config) -> str:
    if not cfg.use_router:
        return "direct"
    if re.search(r"\d", user_input) and any(op in user_input for op in ["+", "-", "*", "/", "**"]):
        return "calculator"
    return "rag" if cfg.use_rag else "direct"

def run_pipeline(user_input: str, cfg: Config) -> PipelineResult:
    tool_trace = []
    warnings = []

    # Input guardrails
    if cfg.guardrails_on:
        refusal = guardrail_check_input(user_input)
        if refusal:
            return PipelineResult(
                answer=refusal,
                sources=[],
                tool_trace=["guardrail->refuse"],
                warnings=[],
                metadata={"ts": now_ts(), "cfg_hash": stable_hash(asdict(cfg))},
            )

    r = route(user_input, cfg)
    tool_trace.append(f"router->{r}")

    retrieved = []
    retrieval_conf = 0.0

    if r == "calculator":
        out = tool_calculate(user_input, cfg)
        answer = f"The result is: {out['result']}"
        tool_trace.append("tool->calculator")

    elif r == "rag":
        out = tool_rag_search(user_input, cfg)
        retrieved = out["hits"]
        retrieval_conf = float(out["confidence"])
        tool_trace.append(f"tool->rag_search (hits={len(retrieved)})")

        if cfg.require_citations and not retrieved:
            warnings.append("No sources retrieved. Answer may be unreliable.")
        if retrieval_conf < cfg.low_confidence_threshold:
            warnings.append("Low retrieval confidence — consider verifying sources or rephrasing.")

        prompt = build_answer_prompt(user_input, retrieved)
        llm_out = call_llm(prompt, cfg)
        answer = llm_out["text"]

    else:
        prompt = build_answer_prompt(user_input, [])
        llm_out = call_llm(prompt, cfg)
        answer = llm_out["text"]

    # Output guardrails
    if cfg.guardrails_on:
        out_refusal = guardrail_check_output(answer)
        if out_refusal:
            tool_trace.append("guardrail->output_block")
            answer = out_refusal

    result = PipelineResult(
        answer=answer,
        sources=[{"title": c["title"], "source": c.get("source",""), "snippet": c["text"][:280]} for c in retrieved],
        tool_trace=tool_trace,
        warnings=warnings,
        metadata={
            "ts": now_ts(),
            "route": r,
            "retrieval_conf": retrieval_conf,
            "cfg_hash": stable_hash(asdict(cfg)),
        }
    )

    append_jsonl(cfg.log_path, {
        "ts": result.metadata["ts"],
        "user_input": user_input,
        "answer": result.answer,
        "sources": result.sources,
        "tool_trace": result.tool_trace,
        "warnings": result.warnings,
        "metadata": result.metadata,
        "cfg": asdict(cfg),
    })

    return result


## 5) Launch Gradio App


In [15]:
import gradio as gr

def gradio_respond(message: str, history: List[Tuple[str, str]]):
    res = run_pipeline(message, cfg)
    history = history + [(message, res.answer)]

    if res.sources:
        sources_md = "\n".join([f"- **{s['title']}** — {s['source']}\n  - {s['snippet']}" for s in res.sources])
    else:
        sources_md = "_No sources retrieved._"

    trace_md = "\n".join([f"- {t}" for t in res.tool_trace]) or "_(none)_"
    warn_md = "\n".join([f"- ⚠️ {w}" for w in res.warnings]) or "_(none)_"

    return history, sources_md, trace_md, warn_md

with gr.Blocks() as demo:
    gr.Markdown("# Project Demo UI (Starter)")
    gr.Markdown("Tip: Ask about **library hours** or **dining hours** to see retrieval in action.")
    chatbot = gr.Chatbot(label="Chat", height=280)
    msg = gr.Textbox(label="Your message", placeholder="Type a question and press Enter...")
    sources = gr.Markdown(label="Sources")
    trace = gr.Markdown(label="Tool trace")
    warnings = gr.Markdown(label="Warnings")

    def clear():
        return [], "", "", ""

    msg.submit(gradio_respond, inputs=[msg, chatbot], outputs=[chatbot, sources, trace, warnings])
    gr.Button("Clear").click(clear, outputs=[chatbot, sources, trace, warnings])

demo


/tmp/ipykernel_22936/4159806487.py:20: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat", height=280)
/tmp/ipykernel_22936/4159806487.py:20: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat", height=280)


Gradio Blocks instance: 2 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7facca68c650>
 |-<gradio.components.chatbot.Chatbot object at 0x7faccbc100e0>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x7faccbc100e0>
 |-<gradio.components.markdown.Markdown object at 0x7facca6be240>
 |-<gradio.components.markdown.Markdown object at 0x7faccc882cf0>
 |-<gradio.components.markdown.Markdown object at 0x7facca7d0440>
fn_index=1
 inputs:
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x7faccbc100e0>
 |-<gradio.components.markdown.Markdown object at 0x7facca6be240>
 |-<gradio.components.markdown.Markdown object at 0x7faccc882cf0>
 |-<gradio.components.markdown.Markdown object at 0x7facca7d0440>

In [16]:
# --- Launch Gradio App ---
# In Colab, you can set share=True to get a public link (if allowed by your course policy).
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c9116fd54293f8a5a6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Some questions to ask

1. **Query:** *"Why was Nico Hulkenbergs collision in the 2025 abu dhabi grand prix given no further action?"*
   - **Ideal Response:** Mentions the decision from the Abu Dhabi Grand Prix for Car 27 and cites the stewards' conclusion. Note: Because of our low chunk size (observed failure mode), it might hallucinate the exact reasoning.

2. **Query:** *"What is the penalty for exceeding the pit lane speed limit?"*
   - **Ideal Response:** Summarizes the regulations related to pit lane speeding and references the decisions applied to Car 23 or Car 18 in Qatar/Abu Dhabi. 

3. **Query:** *"Is there a penalty for changing direction more than once to defend a position?"*


## AI Usage

Used Google's gemini within Antigravity to help parse through PDF's and convert these into a format that can be used by the code. Understood how this worked then wrote the code myself. 